# Ray Train Distributed Patch Prediction Demo

This notebook demonstrates distributed prediction using Ray Train, where each worker processes a mutually exclusive subset of database shards, generates synthetic patch predictions, and synchronizes with a barrier after each round.

In [1]:
import ray
from ray import train
from ray.train import get_context
from ray.train.collective import barrier
import numpy as np
import psycopg
from psycopg.rows import dict_row
import os
import random
import datetime
from db_client import CitusHeadClient, PredPatchStore
import time

# Configuration
NUM_WORKERS = 8  # Set as needed
PATCHES_PER_BATCH = 10000  # Number of patches per batch
TIME_TO_TRAIN_PER_PATCH = 0  # Simulated time to train on each patch (seconds)
TIME_TO_TRAIN_PER_BATCH = TIME_TO_TRAIN_PER_PATCH * PATCHES_PER_BATCH  # Simulated time to train on each batch (seconds)

# DB connection (reuse constants from db_client)
from constants import CITUS_HEAD_HOST, CITUS_HEAD_PORT, CITUS_HEAD_DB, CITUS_HEAD_USER, CITUS_HEAD_PASSWORD

def get_db_conn():
    return psycopg.connect(
        host=CITUS_HEAD_HOST,
        port=CITUS_HEAD_PORT,
        dbname=CITUS_HEAD_DB,
        user=CITUS_HEAD_USER,
        password=CITUS_HEAD_PASSWORD,
        autocommit=False,
        row_factory=dict_row
    )


In [2]:
# Truncate pred_patch_latest, pred_patch_last, and confusion matrix tables at the start (for a clean run)
with get_db_conn() as conn:
    with conn.cursor() as cur:
        try:
            cur.execute("TRUNCATE TABLE pred_patch_latest;")
        except Exception as e:
            print("Warning: pred_patch_latest could not be truncated:", e)
        try:
            cur.execute("TRUNCATE TABLE pred_patch_last;")
        except Exception as e:
            print("Warning: pred_patch_last could not be truncated:", e)
        for lvl in range(8, 13):
            try:
                cur.execute(f"TRUNCATE TABLE confusion_matrix_l{lvl};")
            except Exception as e:
                print(f"Warning: confusion_matrix_l{lvl} could not be truncated:", e)

In [3]:
def get_patch_shards():
    """Return a sorted list of all non-empty patch table shard IDs."""
    with get_db_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT shardid FROM pg_dist_shard WHERE logicalrelid = 'patch'::regclass ORDER BY shardid;")
            return [row['shardid'] for row in cur.fetchall()]


class ShardDataset:
    """Loads real patches from assigned Citus shard tables and yields them in batches."""
    def __init__(self, assigned_shards: list, patches_per_batch: int):
        self.assigned_shards = assigned_shards
        self.patches_per_batch = patches_per_batch

    def __iter__(self):
        client = CitusHeadClient()
        # TODO: optimize this by fetching only the patch IDs and then fetching the full patch data in batches, instead of fetching all patches at once
        patches = client.fetch_patches_by_shards(self.assigned_shards)
        for i in range(0, len(patches), self.patches_per_batch):
            yield patches[i:i + self.patches_per_batch]

In [4]:
# def insert_predictions(batch: list):
#     with get_db_conn() as conn:
#         with conn.cursor() as cur:
#             with cur.copy(
#                 """
#                 COPY pred_patch_latest
#                     (patch_id, embed_x, embed_y, grid_cell_i, grid_cell_j, event_ts, label_class_id)
#                 FROM STDIN WITH (FORMAT TEXT)
#                 """
#             ) as copy:
#                 for patch in batch:
#                     copy.write_row((
#                         patch['patch_id'],
#                         random.uniform(0.0, 1.0),
#                         random.uniform(0.0, 1.0),
#                         random.randint(0, 4096),
#                         random.randint(0, 4096),
#                         datetime.datetime.now(),
#                         int(patch['label_class_id']),
#                     ))

In [5]:
def insert_predictions(batch: list, center: tuple=(0.5, 0.5), stddev: float=0.15):
    with get_db_conn() as conn:
        with conn.cursor() as cur:
            with cur.copy(
                """
                COPY pred_patch_latest
                    (patch_id, embed_x, embed_y, grid_cell_i, grid_cell_j, event_ts, label_class_id)
                FROM STDIN WITH (FORMAT TEXT)
                """
            ) as copy:
                for patch in batch:
                    copy.write_row((
                        patch['patch_id'],
                        random.gauss(center[0], stddev),
                        random.gauss(center[1], stddev),
                        max(0, min(4096, int(random.gauss(4096 * center[0], 128)))),
                        max(0, min(4096, int(random.gauss(4096 * center[1], 128)))),
                        datetime.datetime.now(),
                        int(patch['label_class_id']) + random.choice([0, 1, 2, 3, 4, 5]),  # Simulate some label noise
                    ))

In [6]:
def train_worker(config):
    context = get_context()
    rank = context.get_world_rank()
    world_size = context.get_world_size()
    print(f"[Worker {rank}] Starting. World size: {world_size}")

    all_shards = config['all_shards']
    assigned_shards = [s for i, s in enumerate(all_shards) if i % world_size == rank]
    print(f"[Worker {rank}] Assigned {len(assigned_shards)} shards: {assigned_shards}")

    pred_store = PredPatchStore(CitusHeadClient())

    cycle = 0
    while True:
        cycle += 1
        print(f"[Worker {rank}] Starting cycle {cycle}.")

        dataset = ShardDataset(assigned_shards, config['patches_per_batch'])
        center = (random.uniform(0.25, 0.75), random.uniform(0.25, 0.75))  # Simulate model drift by changing the center each cycle
        num_batches = 0
        for batch_num, batch in enumerate(dataset):
            print(f"[Worker {rank}] Cycle {cycle} — batch {batch_num + 1} ({len(batch)} patches).")
            insert_predictions(batch, center=center)
            time.sleep(TIME_TO_TRAIN_PER_BATCH)
            num_batches += 1
        print(f"[Worker {rank}] Cycle {cycle} done ({num_batches} batches). Waiting at barrier.")

        # Barrier 1: all workers finished inserting for this cycle
        barrier()

        # Rank 0 rotates tables while other workers wait
        if rank == 0:
            pred_store.rotate_tables()
            print("[Rank 0] Table rotation complete: pred_patch_latest is fresh, pred_patch_last holds previous cycle.")

        # Barrier 2: rotation complete, all workers resume
        barrier()
        print(f"[Worker {rank}] Cycle {cycle} rotation complete. Starting next cycle.")


In [7]:
# Main Ray Train execution cell
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig

if not ray.is_initialized():
    ray.init(runtime_env={
        "env_vars": {"RAY_DEBUG": "1"}, 
    })

# Fetch all patch shard IDs for distribution across workers
all_shards = get_patch_shards()
print(f"Total patch shards: {len(all_shards)}")

trainer = TorchTrainer(
    train_loop_per_worker=train_worker,
    train_loop_config={
        "all_shards": all_shards,
        "patches_per_batch": PATCHES_PER_BATCH,
    },
    scaling_config=ScalingConfig(
        num_workers=NUM_WORKERS,
        use_gpu=False,
    ),
)

result = trainer.fit()
print("Ray Train Results:")
print(result)

2026-05-18 12:11:18,574	INFO worker.py:2012 -- Started a local Ray instance.
/Users/jackson/Research/code/histotools/PatchSorter/.venv/lib/python3.14/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Total patch shards: 32


(TrainController pid=13267) Requesting resources: {'CPU': 1} * 8
(TrainController pid=13267) Attempting to start training worker group of size 8 with the following resources: [{'CPU': 1}] * 8
(PlacementGroupCleaner pid=13279) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(RayTrainWorker pid=13276) Setting up process group for: env:// [rank=0, world_size=8]
(PlacementGroupCleaner pid=13279) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(TrainController pid=13267) Started training worker group of size 8: 
(TrainController pid=13267) - (ip=127.0.0.1, pid=13276) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=13267) - (ip=127.0.0.1, pid=13275) world_rank=1, local_rank=1, node_rank=0
(TrainController pid=13267) - (ip=127.0.0.1, pid=13280) world_rank=2, local_rank=2, node_rank=0
(TrainController pid=13267) - (ip=127.0.0.1, pid=13284) world_ran

SystemExit: 1

/Users/jackson/Research/code/histotools/PatchSorter/.venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
